# TP CNN — LeNet-5 sur la base AMHCD (caractères Tifinagh, RGB)

**Auteur : Bilal El Ibrahimi (IAA) — Pr. M. Benaddy, Pr. A. Hadri — 2024-2025**

Ce notebook réimplémente **LeNet-5 entièrement en NumPy** (aucun framework de Deep Learning) :

* toutes les couches sont des classes paramétrables (`Conv2D`, `Pool2D`, `Dense`, `Flatten`, activations) avec `forward` et `backward` écrits à la main ;
* perte *softmax + entropie croisée* ;
* optimiseurs `SGD` (avec moment / Nesterov), `RMSprop`, `Adam` ;
* boucle d'apprentissage train / validation / test avec sauvegarde du meilleur modèle ;
* vérification des gradients par différences finies centrées ;
* visualisations : courbes loss/accuracy, matrice de confusion, cartes caractéristiques, filtres, erreurs.

**Différence avec le TP précédent (Tifinagh niveaux de gris)** : les images AMHCD sont en **RGB**, donc l'entrée est `32 × 32 × 3` et C1 possède `6 × (5×5×3) + 6 = 456` paramètres.

## 1. Configuration

In [ ]:
import os, time, json, glob
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from numpy.lib.stride_tricks import sliding_window_view
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, f1_score, classification_report

SEED        = 42
IMG_SIZE    = 32            # taille d'entrée de LeNet-5
N_CHANNELS  = 3             # RGB
FAST        = os.environ.get("LENET_FAST", "0") == "1"   # mode test rapide (jeu synthétique)
EPOCHS      = int(os.environ.get("LENET_EPOCHS", 30))     # entraînement principal
ABL_EPOCHS  = int(os.environ.get("LENET_ABL_EPOCHS", 12)) # études comparatives
BATCH_SIZE  = 64
DATA_DIR    = os.environ.get("AMHCD_DIR", "/kaggle/input/amazigh-handwritten-character-database-amhcd")
FIG_DIR     = "figures"; os.makedirs(FIG_DIR, exist_ok=True)
np.random.seed(SEED)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .3})
print("FAST =", FAST, "| EPOCHS =", EPOCHS, "| ABL_EPOCHS =", ABL_EPOCHS)

## 2. Chargement de la base AMHCD

La base est organisée en un dossier par classe (33 lettres Tifinagh). Le chargeur :

1. cherche automatiquement les dossiers contenant ≥ 33 sous-dossiers de classes ;
2. lit chaque image en **RGB**, la redimensionne en 32×32, la met à l'échelle dans [0,1] ;
3. découpe en **train / validation / test = 70 / 15 / 15 %** de façon *stratifiée* ;
4. standardise par canal avec la moyenne et l'écart-type **du train uniquement** (pas de fuite d'information).

> Si la base n'est pas trouvée (par exemple lors d'un test hors-ligne), un **jeu synthétique** est généré pour valider le code. Le drapeau `USING_SYNTHETIC` l'indique : **les résultats du rapport doivent être produits avec la vraie base.**

In [ ]:
from PIL import Image
IMG_EXT = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")

def find_class_roots(base, min_classes=30):
    """Retourne les dossiers dont les sous-dossiers (classes) contiennent des images."""
    roots = []
    for dirpath, dirnames, _ in os.walk(base):
        good = [d for d in dirnames
                if any(f.lower().endswith(IMG_EXT) for f in os.listdir(os.path.join(dirpath, d)))]
        if len(good) >= min_classes:
            roots.append(dirpath)
    return roots

def load_images(roots, size=IMG_SIZE):
    classes = sorted({d for r in roots for d in os.listdir(r) if os.path.isdir(os.path.join(r, d))})
    cidx = {c: i for i, c in enumerate(classes)}
    X, y = [], []
    for r in roots:
        for c in sorted(os.listdir(r)):
            p = os.path.join(r, c)
            if not os.path.isdir(p) or c not in cidx: continue
            for f in sorted(os.listdir(p)):
                if f.lower().endswith(IMG_EXT):
                    im = Image.open(os.path.join(p, f)).convert("RGB").resize((size, size), Image.BILINEAR)
                    X.append(np.asarray(im, dtype=np.float32) / 255.0); y.append(cidx[c])
    return np.stack(X).transpose(0, 3, 1, 2), np.array(y), classes   # NCHW

def make_synthetic(n_classes=33, per_class=40, size=IMG_SIZE, seed=0):
    """Jeu factice (prototypes 8x8 agrandis + bruit + décalage) pour tester le code hors-ligne."""
    rng = np.random.default_rng(seed)
    protos = rng.random((n_classes, 3, 8, 8)).repeat(4, axis=2).repeat(4, axis=3)
    X, y = [], []
    for c in range(n_classes):
        for _ in range(per_class):
            im = np.roll(protos[c], rng.integers(-2, 3, 2), axis=(1, 2)) + 0.25 * rng.standard_normal(protos[c].shape)
            X.append(np.clip(im, 0, 1)); y.append(c)
    return np.array(X, dtype=np.float32), np.array(y), [f"c{c:02d}" for c in range(n_classes)]

USING_SYNTHETIC = False
roots = find_class_roots(DATA_DIR) if os.path.isdir(DATA_DIR) else []
if not roots:
    try:                                   # Colab / local : tentative kagglehub
        import kagglehub
        DATA_DIR = kagglehub.dataset_download("benaddym/amazigh-handwritten-character-database-amhcd")
        roots = find_class_roots(DATA_DIR)
    except Exception as e:
        print("kagglehub indisponible :", type(e).__name__)
if roots:
    print("Dossiers de classes trouvés :", roots)
    X, y, CLASSES = load_images(roots)
else:
    USING_SYNTHETIC = True
    print("!!! BASE AMHCD INTROUVABLE -> jeu SYNTHÉTIQUE (test du code uniquement) !!!")
    X, y, CLASSES = make_synthetic()
N_CLASSES = len(CLASSES)
print(f"X: {X.shape} (N,C,H,W), y: {y.shape}, classes: {N_CLASSES}, synthétique: {USING_SYNTHETIC}")

In [ ]:
Xtr, Xtmp, ytr, ytmp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=SEED)
Xva, Xte, yva, yte   = train_test_split(Xtmp, ytmp, test_size=0.50, stratify=ytmp, random_state=SEED)
mean = Xtr.mean(axis=(0, 2, 3), keepdims=True); std = Xtr.std(axis=(0, 2, 3), keepdims=True) + 1e-8
Xtr_n, Xva_n, Xte_n = [((a - mean) / std).astype(np.float32) for a in (Xtr, Xva, Xte)]
print("train/val/test :", len(Xtr), len(Xva), len(Xte))
print("effectif par classe (min/max) dans train :", np.bincount(ytr).min(), np.bincount(ytr).max())

# Aperçu : une image par classe
fig, axs = plt.subplots(3, 11, figsize=(14, 5))
for k, ax in enumerate(axs.ravel()):
    if k < N_CLASSES:
        i = np.where(y == k)[0][0]; ax.imshow(X[i].transpose(1, 2, 0)); ax.set_title(CLASSES[k], fontsize=8)
    ax.axis("off")
plt.suptitle("Aperçu de la base (une image par classe)"); plt.tight_layout()
plt.savefig(f"{FIG_DIR}/apercu_base.png"); plt.show()

## 3. Couches paramétrables (forward / backward)

**Convolution** (stride 1, padding $p$) pour une entrée $x\in\mathbb{R}^{N\times C\times H\times W}$ et des filtres $W\in\mathbb{R}^{F\times C\times k\times k}$ :

$$y_{n,f,i,j}=b_f+\sum_{c=1}^{C}\sum_{a=0}^{k-1}\sum_{b=0}^{k-1} W_{f,c,a,b}\;x_{n,c,i+a,j+b},\qquad H_{out}=H+2p-k+1 .$$

**Rétropropagation** (avec $\delta=\partial\mathcal L/\partial y$) :

$$\frac{\partial\mathcal L}{\partial W_{f,c,a,b}}=\sum_{n,i,j}\delta_{n,f,i,j}\,x_{n,c,i+a,j+b},\quad
\frac{\partial\mathcal L}{\partial b_f}=\sum_{n,i,j}\delta_{n,f,i,j},\quad
\frac{\partial\mathcal L}{\partial x_{n,c,i,j}}=\sum_{f,a,b}\delta_{n,f,i-a,j-b}\,W_{f,c,a,b}$$

(la dernière est une convolution « pleine » de $\delta$ avec le filtre retourné). Le tout est vectorisé avec `sliding_window_view` + `tensordot`.

**Pooling moyen** $k\times k$, stride $k$ : $y=\frac1{k^2}\sum x$ ; gradient réparti uniformément $\delta/k^2$. **Pooling max** : le gradient est transmis à l'élément maximal.

**Couche dense** : $y=xW+b$ ; $\partial\mathcal L/\partial W=x^\top\delta$, $\partial\mathcal L/\partial b=\sum_n\delta$, $\partial\mathcal L/\partial x=\delta W^\top$.

**Activations** : $\tanh'(z)=1-\tanh^2 z$ ; $\mathrm{ReLU}'(z)=\mathbb 1_{z>0}$ ; $\sigma'(z)=\sigma(z)(1-\sigma(z))$.

In [ ]:
class Layer:
    """Classe de base : une couche possède des paramètres (params) et leurs gradients (grads)."""
    name = "layer"
    def __init__(self): self.params, self.grads = {}, {}
    def forward(self, x, train=True): raise NotImplementedError
    def backward(self, dout): raise NotImplementedError

def init_weights(shape, fan_in, fan_out, scheme, rng, dtype):
    if scheme == "xavier":  s = np.sqrt(2.0 / (fan_in + fan_out))
    elif scheme == "he":    s = np.sqrt(2.0 / fan_in)
    elif scheme == "lecun": s = np.sqrt(1.0 / fan_in)
    else: raise ValueError(scheme)
    return (rng.standard_normal(shape) * s).astype(dtype)

class Conv2D(Layer):
    """Convolution 2D, stride 1, padding paramétrable. Entrée/sortie au format NCHW."""
    def __init__(self, in_ch, out_ch, k=5, padding=0, init="xavier", rng=None, dtype=np.float32, name="conv"):
        super().__init__(); rng = rng or np.random.default_rng(0)
        assert in_ch > 0 and out_ch > 0 and k > 0 and padding >= 0
        self.k, self.p, self.name = k, padding, name
        self.params["W"] = init_weights((out_ch, in_ch, k, k), in_ch * k * k, out_ch * k * k, init, rng, dtype)
        self.params["b"] = np.zeros(out_ch, dtype=dtype)
    def forward(self, x, train=True):
        assert x.ndim == 4 and x.shape[1] == self.params["W"].shape[1], f"entrée inattendue {x.shape}"
        p, k = self.p, self.k
        xp = np.pad(x, ((0, 0), (0, 0), (p, p), (p, p))) if p else x
        self.win = sliding_window_view(xp, (k, k), axis=(2, 3))            # (N,C,Ho,Wo,k,k)
        out = np.tensordot(self.win, self.params["W"], axes=([1, 4, 5], [1, 2, 3]))  # (N,Ho,Wo,F)
        return out.transpose(0, 3, 1, 2) + self.params["b"][None, :, None, None]
    def backward(self, dout):
        k, p = self.k, self.p
        self.grads["W"] = np.tensordot(dout, self.win, axes=([0, 2, 3], [0, 2, 3]))
        self.grads["b"] = dout.sum(axis=(0, 2, 3))
        dpad = np.pad(dout, ((0, 0), (0, 0), (k - 1, k - 1), (k - 1, k - 1)))
        wd = sliding_window_view(dpad, (k, k), axis=(2, 3))                 # (N,F,Hp,Wp,k,k)
        dxp = np.tensordot(wd, self.params["W"][:, :, ::-1, ::-1], axes=([1, 4, 5], [0, 2, 3]))
        dx = dxp.transpose(0, 3, 1, 2)
        return dx[:, :, p:dx.shape[2] - p, p:dx.shape[3] - p] if p else dx

class Pool2D(Layer):
    """Pooling k×k, stride = k, mode 'avg' (LeNet-5 original) ou 'max'."""
    def __init__(self, k=2, mode="avg", name="pool"):
        super().__init__(); assert mode in ("avg", "max"); self.k, self.mode, self.name = k, mode, name
    def forward(self, x, train=True):
        N, C, H, W = x.shape; k = self.k
        assert H % k == 0 and W % k == 0, "H et W doivent être divisibles par k"
        xr = x.reshape(N, C, H // k, k, W // k, k)
        if self.mode == "avg": return xr.mean(axis=(3, 5))
        out = xr.max(axis=(3, 5)); self.mask = (xr == out[:, :, :, None, :, None]); self.xshape = x.shape
        return out
    def backward(self, dout):
        k = self.k; N, C, Ho, Wo = dout.shape
        if self.mode == "avg":
            return (dout[:, :, :, None, :, None] / k**2 * np.ones((1, 1, 1, k, 1, k), dout.dtype)).reshape(N, C, Ho * k, Wo * k)
        m = self.mask / self.mask.sum(axis=(3, 5), keepdims=True)          # gestion des égalités
        return (m * dout[:, :, :, None, :, None]).reshape(N, C, Ho * k, Wo * k)

class Flatten(Layer):
    name = "flatten"
    def forward(self, x, train=True): self.shape = x.shape; return x.reshape(x.shape[0], -1)
    def backward(self, dout): return dout.reshape(self.shape)

class Dense(Layer):
    """Couche entièrement connectée y = xW + b (C5, F6 et sortie)."""
    def __init__(self, n_in, n_out, init="xavier", rng=None, dtype=np.float32, name="fc"):
        super().__init__(); rng = rng or np.random.default_rng(0); self.name = name
        self.params["W"] = init_weights((n_in, n_out), n_in, n_out, init, rng, dtype)
        self.params["b"] = np.zeros(n_out, dtype=dtype)
    def forward(self, x, train=True):
        assert x.ndim == 2 and x.shape[1] == self.params["W"].shape[0], f"entrée inattendue {x.shape}"
        self.x = x; return x @ self.params["W"] + self.params["b"]
    def backward(self, dout):
        self.grads["W"] = self.x.T @ dout; self.grads["b"] = dout.sum(axis=0)
        return dout @ self.params["W"].T

class Activation(Layer):
    def __init__(self, kind="tanh", name=None):
        super().__init__(); assert kind in ("tanh", "relu", "sigmoid"); self.kind = kind; self.name = name or kind
    def forward(self, x, train=True):
        if self.kind == "tanh": self.out = np.tanh(x)
        elif self.kind == "relu": self.out = np.maximum(x, 0)
        else: self.out = 1.0 / (1.0 + np.exp(-x))
        return self.out
    def backward(self, dout):
        if self.kind == "tanh": return dout * (1 - self.out**2)
        if self.kind == "relu": return dout * (self.out > 0)
        return dout * self.out * (1 - self.out)

def softmax_cross_entropy(logits, y):
    """Perte moyenne L = -(1/N) Σ log p_{n,y_n} et gradient dL/dlogits = (p - onehot)/N."""
    z = logits - logits.max(axis=1, keepdims=True)
    e = np.exp(z); p = e / e.sum(axis=1, keepdims=True)
    N = len(y); loss = -np.log(p[np.arange(N), y] + 1e-12).mean()
    d = p.copy(); d[np.arange(N), y] -= 1
    return float(loss), d / N

## 4. Réseau LeNet-5 (modèle séquentiel paramétrable)

| Couche | Opération | Sortie | Paramètres (RGB) |
|---|---|---|---|
| Entrée | image RGB | 3@32×32 | 0 |
| C1 | conv 5×5, 6 filtres + act. | 6@28×28 | 456 |
| S2 | pooling moyen 2×2 | 6@14×14 | 0 |
| C3 | conv 5×5, 16 filtres + act. | 16@10×10 | 2 416 |
| S4 | pooling moyen 2×2 | 16@5×5 | 0 |
| C5 | dense 400→120 + act. | 120 | 48 120 |
| F6 | dense 120→84 + act. | 84 | 10 164 |
| Sortie | dense 84→33 + softmax | 33 | 2 805 |

Total : **63 961 paramètres**.

In [ ]:
class Sequential:
    def __init__(self, layers): self.layers = layers
    def forward(self, x, train=True, collect=False):
        acts = []
        for l in self.layers:
            x = l.forward(x, train)
            if collect: acts.append((l.name, x))
        return (x, acts) if collect else x
    def backward(self, d):
        for l in reversed(self.layers): d = l.backward(d)
        return d
    def parameters(self):
        for i, l in enumerate(self.layers):
            for n in l.params: yield (i, n), l.params[n], l.grads.get(n)
    def n_params(self): return sum(p.size for _, p, _ in self.parameters())
    def get_weights(self): return {k: p.copy() for k, p, _ in self.parameters()}
    def set_weights(self, w):
        for k, p, _ in self.parameters(): p[...] = w[k]

def build_lenet5(in_ch=N_CHANNELS, n_classes=N_CLASSES, filters=(6, 16), k=5, fc=(120, 84),
                 act="tanh", pool="avg", init="xavier", img=IMG_SIZE, seed=SEED, dtype=np.float32):
    """LeNet-5 paramétrable. Pour img=32, k=5 : C1 28x28 -> S2 14x14 -> C3 10x10 -> S4 5x5."""
    rng = np.random.default_rng(seed)
    s = (img - k + 1) // 2; s = (s - k + 1) // 2
    return Sequential([
        Conv2D(in_ch, filters[0], k, 0, init, rng, dtype, "C1"), Activation(act, "C1_act"), Pool2D(2, pool, "S2"),
        Conv2D(filters[0], filters[1], k, 0, init, rng, dtype, "C3"), Activation(act, "C3_act"), Pool2D(2, pool, "S4"),
        Flatten(),
        Dense(filters[1] * s * s, fc[0], init, rng, dtype, "C5"), Activation(act, "C5_act"),
        Dense(fc[0], fc[1], init, rng, dtype, "F6"), Activation(act, "F6_act"),
        Dense(fc[1], n_classes, init, rng, dtype, "OUT")])

net = build_lenet5()
_, acts = net.forward(Xtr_n[:2], collect=True)
for n, a in acts: print(f"{n:8s} -> {a.shape}")
print("Nombre de paramètres :", net.n_params())
assert net.n_params() == 63961 or N_CHANNELS != 3 or N_CLASSES != 33

## 5. Vérification des gradients (différences finies centrées)

$$\frac{\partial\mathcal L}{\partial\theta}\approx\frac{\mathcal L(\theta+\varepsilon)-\mathcal L(\theta-\varepsilon)}{2\varepsilon},\qquad
\text{erreur relative}=\frac{\lVert g_{ana}-g_{num}\rVert}{\lVert g_{ana}\rVert+\lVert g_{num}\rVert}$$

Test en **float64**, pour les deux modes de pooling et deux activations, y compris avec padding.

In [ ]:
def grad_check(model, x, y, n_check=6, eps=1e-5, seed=0):
    rng = np.random.default_rng(seed)
    loss, d = softmax_cross_entropy(model.forward(x), y); model.backward(d)
    worst = 0.0
    for key, p, g in model.parameters():
        idxs = [tuple(rng.integers(0, s) for s in p.shape) for _ in range(n_check)]
        ga, gn = [], []
        for ix in idxs:
            old = p[ix]
            p[ix] = old + eps; lp, _ = softmax_cross_entropy(model.forward(x), y)
            p[ix] = old - eps; lm, _ = softmax_cross_entropy(model.forward(x), y)
            p[ix] = old; ga.append(g[ix]); gn.append((lp - lm) / (2 * eps))
        ga, gn = np.array(ga), np.array(gn)
        worst = max(worst, np.linalg.norm(ga - gn) / (np.linalg.norm(ga) + np.linalg.norm(gn) + 1e-15))
    return worst

xg = np.random.default_rng(1).standard_normal((3, 3, 32, 32)); yg = np.array([0, 5, 7])
for act in ("tanh", "relu", "sigmoid"):
    for pool in ("avg", "max"):
        m = build_lenet5(act=act, pool=pool, dtype=np.float64, seed=3)
        e = grad_check(m, xg, yg); print(f"act={act:7s} pool={pool}  erreur relative max = {e:.2e}")
        assert e < 1e-4, "gradient incorrect"
# test supplémentaire : convolution avec padding (LeNet « same » : C1 avec p=2)
m = Sequential([Conv2D(3, 4, 5, 2, "xavier", np.random.default_rng(0), np.float64), Activation("tanh"), Pool2D(2, "avg"),
                Flatten(), Dense(4 * 16 * 16, 33, "xavier", np.random.default_rng(1), np.float64)])
e = grad_check(m, xg, yg); print(f"conv avec padding=2 : erreur relative max = {e:.2e}"); assert e < 1e-4
print("OK : tous les gradients sont corrects.")

## 6. Optimiseurs

Avec $g_t=\nabla_\theta\mathcal L$ et décroissance des poids $\lambda$ (régularisation $L_2$, ajoutée au gradient) :

* **SGD** : $\theta\leftarrow\theta-\eta\,g$ ; **avec moment** : $v\leftarrow\mu v+g,\ \theta\leftarrow\theta-\eta v$ (Nesterov : $\theta\leftarrow\theta-\eta(g+\mu v)$).
* **RMSprop** : $s\leftarrow\rho s+(1-\rho)g^2,\ \theta\leftarrow\theta-\eta\,g/(\sqrt s+\epsilon)$.
* **Adam** : $m\leftarrow\beta_1m+(1-\beta_1)g,\ v\leftarrow\beta_2v+(1-\beta_2)g^2,\ \hat m=\frac{m}{1-\beta_1^t},\ \hat v=\frac{v}{1-\beta_2^t},\ \theta\leftarrow\theta-\eta\frac{\hat m}{\sqrt{\hat v}+\epsilon}$.

In [ ]:
class Optimizer:
    def __init__(self, lr, weight_decay=0.0): self.lr, self.wd, self.t = lr, weight_decay, 0
    def step(self, model):
        self.t += 1
        for key, p, g in model.parameters():
            if self.wd and p.ndim > 1: g = g + self.wd * p          # pas de décroissance sur les biais
            self.update(key, p, g)
    def update(self, key, p, g): raise NotImplementedError

class SGD(Optimizer):
    def __init__(self, lr=0.01, momentum=0.0, nesterov=False, weight_decay=0.0):
        super().__init__(lr, weight_decay); self.mu, self.nesterov, self.v = momentum, nesterov, {}
    def update(self, key, p, g):
        if self.mu == 0: p -= self.lr * g; return
        v = self.v.setdefault(key, np.zeros_like(p)); v *= self.mu; v += g
        p -= self.lr * ((g + self.mu * v) if self.nesterov else v)

class RMSprop(Optimizer):
    def __init__(self, lr=1e-3, rho=0.9, eps=1e-8, weight_decay=0.0):
        super().__init__(lr, weight_decay); self.rho, self.eps, self.s = rho, eps, {}
    def update(self, key, p, g):
        s = self.s.setdefault(key, np.zeros_like(p)); s *= self.rho; s += (1 - self.rho) * g * g
        p -= self.lr * g / (np.sqrt(s) + self.eps)

class Adam(Optimizer):
    def __init__(self, lr=1e-3, b1=0.9, b2=0.999, eps=1e-8, weight_decay=0.0):
        super().__init__(lr, weight_decay); self.b1, self.b2, self.eps, self.m, self.v = b1, b2, eps, {}, {}
    def update(self, key, p, g):
        m = self.m.setdefault(key, np.zeros_like(p)); v = self.v.setdefault(key, np.zeros_like(p))
        m *= self.b1; m += (1 - self.b1) * g; v *= self.b2; v += (1 - self.b2) * g * g
        mh = m / (1 - self.b1 ** self.t); vh = v / (1 - self.b2 ** self.t)
        p -= self.lr * mh / (np.sqrt(vh) + self.eps)

# Test rapide : chaque optimiseur doit réduire la perte sur un mini-lot
for name, opt in [("SGD", SGD(0.05)), ("SGD+mom", SGD(0.02, 0.9)), ("Nesterov", SGD(0.02, 0.9, True)),
                  ("RMSprop", RMSprop(1e-3)), ("Adam", Adam(1e-3))]:
    m = build_lenet5(seed=1); xb, yb = Xtr_n[:32], ytr[:32]
    l0, _ = softmax_cross_entropy(m.forward(xb), yb)
    for _ in range(30):
        l, d = softmax_cross_entropy(m.forward(xb), yb); m.backward(d); opt.step(m)
    print(f"{name:9s} perte {l0:.3f} -> {l:.3f}"); assert l < l0

## 7. Fonction d'apprentissage (train / validation / test)

À chaque époque : mini-lots mélangés, rétropropagation, mise à jour ; puis évaluation sur la validation. Le **meilleur modèle selon l'accuracy de validation** est conservé et utilisé pour le test final (le test n'est jamais utilisé pour choisir quoi que ce soit).

In [ ]:
def evaluate(model, X, y, bs=256):
    losses, preds = 0.0, []
    for i in range(0, len(X), bs):
        lg = model.forward(X[i:i + bs], train=False)
        losses += softmax_cross_entropy(lg, y[i:i + bs])[0] * len(lg); preds.append(lg.argmax(1))
    preds = np.concatenate(preds)
    return losses / len(X), float((preds == y).mean()), preds

def fit(model, opt, Xtr, ytr, Xva, yva, epochs=20, batch_size=BATCH_SIZE, lr_decay=1.0, seed=SEED, verbose=True):
    rng = np.random.default_rng(seed); n = len(Xtr)
    H = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "time": []}
    best_acc, best_w, best_ep = -1, None, 0
    for ep in range(1, epochs + 1):
        t0 = time.time(); perm = rng.permutation(n); tot, ok = 0.0, 0
        for i in range(0, n, batch_size):
            idx = perm[i:i + batch_size]
            lg = model.forward(Xtr[idx], train=True)
            loss, d = softmax_cross_entropy(lg, ytr[idx]); model.backward(d); opt.step(model)
            tot += loss * len(idx); ok += (lg.argmax(1) == ytr[idx]).sum()
        vl, va, _ = evaluate(model, Xva, yva)
        H["train_loss"].append(tot / n); H["train_acc"].append(ok / n)
        H["val_loss"].append(vl); H["val_acc"].append(va); H["time"].append(time.time() - t0)
        if va > best_acc: best_acc, best_w, best_ep = va, model.get_weights(), ep
        opt.lr *= lr_decay
        if verbose and (ep == 1 or ep % max(1, epochs // 10) == 0 or ep == epochs):
            print(f"ép {ep:3d}/{epochs} | loss {tot/n:.4f} acc {ok/n:.4f} | val loss {vl:.4f} acc {va:.4f} | {H['time'][-1]:.1f}s")
    model.set_weights(best_w); H["best_epoch"], H["best_val_acc"] = best_ep, best_acc
    return H

## 8. Entraînement principal : LeNet-5 (tanh, pooling moyen, Adam)

In [ ]:
if FAST: EPOCHS, ABL_EPOCHS = min(EPOCHS, 8), min(ABL_EPOCHS, 4)
main_net = build_lenet5(seed=SEED)
main_opt = Adam(lr=1e-3, weight_decay=1e-4)
t0 = time.time()
H_main = fit(main_net, main_opt, Xtr_n, ytr, Xva_n, yva, epochs=EPOCHS, lr_decay=0.97)
print(f"\nMeilleure époque : {H_main['best_epoch']} (val acc = {H_main['best_val_acc']:.4f}) | durée totale {time.time()-t0:.0f}s")

## 9. Résultats quantitatifs : courbes, test, matrice de confusion

In [ ]:
def plot_curves(H, title, path=None):
    ep = np.arange(1, len(H["train_loss"]) + 1)
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
    ax[0].plot(ep, H["train_loss"], label="train"); ax[0].plot(ep, H["val_loss"], label="validation")
    ax[0].set(xlabel="époque", ylabel="perte (entropie croisée)", title="Loss"); ax[0].legend()
    ax[1].plot(ep, H["train_acc"], label="train"); ax[1].plot(ep, H["val_acc"], label="validation")
    ax[1].axvline(H["best_epoch"], ls="--", c="gray", lw=1)
    ax[1].set(xlabel="époque", ylabel="accuracy", title="Accuracy"); ax[1].legend()
    plt.suptitle(title); plt.tight_layout()
    if path: plt.savefig(path)
    plt.show()

plot_curves(H_main, "LeNet-5 / AMHCD (RGB) — Adam", f"{FIG_DIR}/courbes_loss_acc.png")

tr_l, tr_a, _ = evaluate(main_net, Xtr_n, ytr); va_l, va_a, _ = evaluate(main_net, Xva_n, yva)
te_l, te_a, te_pred = evaluate(main_net, Xte_n, yte)
f1m = f1_score(yte, te_pred, average="macro")
print(f"TRAIN : loss {tr_l:.4f} acc {tr_a:.4f}\nVAL   : loss {va_l:.4f} acc {va_a:.4f}\nTEST  : loss {te_l:.4f} acc {te_a:.4f} | F1 macro {f1m:.4f}")

In [ ]:
cm = confusion_matrix(yte, te_pred, labels=range(N_CLASSES))
cmn = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
fig, ax = plt.subplots(figsize=(10, 9))
im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1); ax.grid(False)
ax.set_xticks(range(N_CLASSES)); ax.set_yticks(range(N_CLASSES))
ax.set_xticklabels(CLASSES, rotation=90, fontsize=7); ax.set_yticklabels(CLASSES, fontsize=7)
ax.set(xlabel="classe prédite", ylabel="classe réelle", title=f"Matrice de confusion normalisée (test, acc = {te_a:.3f})")
plt.colorbar(im, fraction=0.046); plt.tight_layout(); plt.savefig(f"{FIG_DIR}/matrice_confusion.png"); plt.show()

# Classes les plus difficiles et paires les plus confondues
pc_acc = np.diag(cmn); worst = np.argsort(pc_acc)[:5]
print("5 classes les moins bien reconnues :", [(CLASSES[i], round(float(pc_acc[i]), 3)) for i in worst])
off = cm.copy(); np.fill_diagonal(off, 0)
pairs = np.dstack(np.unravel_index(np.argsort(-off, axis=None)[:5], off.shape))[0]
print("Paires les plus confondues (réel -> prédit, n) :", [(CLASSES[a], CLASSES[b], int(off[a, b])) for a, b in pairs])
print(classification_report(yte, te_pred, target_names=CLASSES, digits=3, zero_division=0))

## 10. Résultats qualitatifs : cartes caractéristiques, filtres, erreurs

Les cartes caractéristiques sont les sorties des couches **C1** (après activation), **S2**, **C3** et **S4** pour une même image de test.

In [ ]:
def show_feature_maps(model, x_norm, x_rgb, label, path=None, max_maps=16):
    _, acts = model.forward(x_norm[None], train=False, collect=True)
    d = dict(acts); layers = ["C1_act", "S2", "C3_act", "S4"]
    fig = plt.figure(figsize=(14, 2.1 * len(layers) + 1.6))
    gs = fig.add_gridspec(len(layers) + 1, max_maps, hspace=0.5, wspace=0.1)
    ax = fig.add_subplot(gs[0, :2]); ax.imshow(x_rgb.transpose(1, 2, 0)); ax.set_title(f"entrée ({label})", fontsize=9); ax.axis("off")
    for r, ln in enumerate(layers, 1):
        fm = d[ln][0]
        for j in range(min(max_maps, fm.shape[0])):
            a = fig.add_subplot(gs[r, j]); a.imshow(fm[j], cmap="viridis"); a.axis("off")
            if j == 0: a.set_title(f"{ln} {fm.shape}", fontsize=8, loc="left")
    if path: plt.savefig(path, bbox_inches="tight")
    plt.show()

i = 0
show_feature_maps(main_net, Xte_n[i], Xte[i], CLASSES[yte[i]], f"{FIG_DIR}/cartes_caracteristiques.png")

# Filtres de C1 (6 filtres 5x5x3 affichés en RGB)
W1 = main_net.layers[0].params["W"]
fig, axs = plt.subplots(1, 6, figsize=(10, 2))
for f, a in enumerate(axs):
    w = W1[f].transpose(1, 2, 0); a.imshow((w - w.min()) / (w.max() - w.min() + 1e-8)); a.axis("off"); a.set_title(f"filtre {f+1}", fontsize=8)
plt.suptitle("Filtres appris de C1"); plt.tight_layout(); plt.savefig(f"{FIG_DIR}/filtres_C1.png"); plt.show()

# Exemples mal classés
wrong = np.where(te_pred != yte)[0]; print(f"{len(wrong)} erreurs sur {len(yte)} images de test")
if len(wrong):
    sel = wrong[:12]; fig, axs = plt.subplots(2, 6, figsize=(11, 4.2))
    for a, j in zip(axs.ravel(), sel):
        a.imshow(Xte[j].transpose(1, 2, 0)); a.set_title(f"réel {CLASSES[yte[j]]}\nprédit {CLASSES[te_pred[j]]}", fontsize=8, color="crimson"); a.axis("off")
    for a in axs.ravel()[len(sel):]: a.axis("off")
    plt.suptitle("Exemples mal classés (test)"); plt.tight_layout(); plt.savefig(f"{FIG_DIR}/erreurs.png"); plt.show()

## 11. Études comparatives (bonus)

Mêmes données, même graine, **même nombre d'époques** (`ABL_EPOCHS`) pour comparer : optimiseurs (SGD, SGD+moment, Adam), activation (tanh vs ReLU) et pooling (moyen vs max). La sélection se fait sur la **validation** ; le test est rapporté à titre indicatif.

In [ ]:
configs = {
    "Adam / tanh / avg (référence)": dict(opt=lambda: Adam(1e-3, weight_decay=1e-4), act="tanh", pool="avg"),
    "SGD / tanh / avg":              dict(opt=lambda: SGD(0.05), act="tanh", pool="avg"),
    "SGD+moment / tanh / avg":       dict(opt=lambda: SGD(0.01, 0.9), act="tanh", pool="avg"),
    "RMSprop / tanh / avg":          dict(opt=lambda: RMSprop(5e-4), act="tanh", pool="avg"),
    "Adam / ReLU / avg":             dict(opt=lambda: Adam(1e-3, weight_decay=1e-4), act="relu", pool="avg", init="he"),
    "Adam / tanh / max":             dict(opt=lambda: Adam(1e-3, weight_decay=1e-4), act="tanh", pool="max"),
}
ABL = {}
for name, cfg in configs.items():
    m = build_lenet5(act=cfg["act"], pool=cfg["pool"], init=cfg.get("init", "xavier"), seed=SEED)
    H = fit(m, cfg["opt"](), Xtr_n, ytr, Xva_n, yva, epochs=ABL_EPOCHS, lr_decay=0.97, verbose=False)
    _, ta, tp = evaluate(m, Xte_n, yte)
    ABL[name] = dict(H=H, val=H["best_val_acc"], test=ta, f1=f1_score(yte, tp, average="macro"), t=float(np.sum(H["time"])))
    print(f"{name:32s} val {H['best_val_acc']:.4f} | test {ta:.4f} | F1 {ABL[name]['f1']:.4f} | {ABL[name]['t']:.0f}s")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for name, r in ABL.items():
    ax[0].plot(r["H"]["val_loss"], label=name); ax[1].plot(r["H"]["val_acc"], label=name)
ax[0].set(title="Loss de validation", xlabel="époque"); ax[1].set(title="Accuracy de validation", xlabel="époque")
ax[1].legend(fontsize=7); plt.tight_layout(); plt.savefig(f"{FIG_DIR}/comparaison.png"); plt.show()

## 12. Synthèse et sauvegarde des résultats

In [ ]:
summary = {
    "donnees_synthetiques": USING_SYNTHETIC, "n_classes": N_CLASSES, "n_params": int(main_net.n_params()),
    "split": {"train": len(Xtr), "val": len(Xva), "test": len(Xte)},
    "epochs": EPOCHS, "batch_size": BATCH_SIZE, "best_epoch": H_main["best_epoch"],
    "train_acc": tr_a, "val_acc": va_a, "test_acc": te_a, "test_loss": te_l, "test_f1_macro": float(f1m),
    "temps_total_s": float(np.sum(H_main["time"])),
    "ablation": {k: {kk: float(vv) for kk, vv in v.items() if kk != "H"} for k, v in ABL.items()},
}
json.dump(summary, open("results.json", "w"), indent=2, ensure_ascii=False)
np.savez("lenet5_weights.npz", **{f"{k[0]}_{k[1]}": p for k, p, _ in main_net.parameters()})
print(json.dumps(summary, indent=2, ensure_ascii=False))
if USING_SYNTHETIC: print("\n/!\\ Résultats obtenus sur données SYNTHÉTIQUES : à ne PAS reporter dans l'article.")